In [1]:
import pandas as pd
import numpy as np


## 1. Préparation de l'environnement et des données ESS

On commence par charger la liste de référence de l'ESS. Ce fichier est notre filtre premier : il nous permet d'identifier qui appartient à l'économie sociale et solidaire.

In [ ]:

# Chargement de la liste ESS (on ne garde que le SIREN pour la jointure)
df_ess_ref = pd.read_csv('..\\..\\data\\raw\\entreprisesess.csv', usecols=['SIREN'], dtype={'siren': str})
ess_sirens = set(df_ess_ref['SIREN'])

print(f"Nombre de structures ESS référencées : {len(ess_sirens)}")

Nombre de structures ESS référencées : 1130252


In [ ]:
file = pq.ParquetFile('..\\..\\data\\raw\\StockUniteLegale_utf8.parquet')

## 2. Lecture optimisée du stock INSEE (Parquet)

Pour ne pas charger 30 Millions de lignes, nous allons utiliser une lecture sélective. Choix :  100 000 lignes aléatoires. Nous allons charger un échantillon, mais en ne sélectionnant que les colonnes utiles à la cartographie (SIREN, NAF, Effectifs, etc.).

In [ ]:
# Liste des colonnes nécessaires pour la cartographie
cols_to_keep = [
    'siren', 'denominationUniteLegale', 'nomUniteLegale', 
    'activitePrincipaleUniteLegale', 'trancheEffectifsUniteLegale'
]


J'utilise ici pq.ParquetFile().read_row_group() qui ne charge pas tout le .parquet

In [13]:
import pyarrow.parquet as pq
import random

file = pq.ParquetFile('C:\\Users\\Utilisateur\\OneDrive\\Documents\\centrale Lille + sc po\\Projet SOGA\\ess-numerique-database\\data\\raw\\StockUniteLegale_utf8.parquet')

# Choisir un row group au hasard
rg_index = random.randint(0, file.num_row_groups - 1)

table = file.read_row_group(rg_index, columns=cols_to_keep)
df_chunk = table.to_pandas()
n_sample = min(200000, len(df_chunk)) #je veux le max d'entreprises possibles, mais pas plus de 200k mais ca depend de la taille du row group
# Puis échantillonner dans ce chunk
df_sample = df_chunk.sample(n=n_sample, random_state=42)

print(len(df_sample))


123555


## 3. Le "Double Filtre" : ESS et Numérique

Nous appliquons deux filtres successifs :

    Filtre ESS : Est-ce que le SIREN est présent dans le fichier entreprisesess.csv ?

    Filtre sectoriel : Est-ce que le code NAF correspond aux activités numériques ?

In [ ]:
codes_naf_numerique = [
    '62.01Z', '62.02A', '62.02B', '62.03Z', 
    '63.11Z', '63.12Z', '58.21Z', '58.29C', '58.29A', '58.29B'
]

In [14]:
df_ess_only = df_sample[df_sample['siren'].isin(ess_sirens)].copy()

In [16]:
df_ess_only.head()

,siren,denominationUniteLegale,nomUniteLegale,activitePrincipaleUniteLegale,trancheEffectifsUniteLegale


On remarque ici qu'en prenant 1 row group avec 123000 lignes nous n'avons aucune structure ESS. C'est en fait assez probable, j'ai l'impression que les row group ne sont pas du tout aléatoires. Dans le 02, je vais essayer différemment